# Notebook 01: AI Coding Workflow

## How to Use AI Coding Assistants Effectively

This notebook covers practical patterns for using AI coding assistants in your daily workflow. You'll learn prompt patterns, see before/after examples, and practice debugging and code review with AI.

## 1. The AI Coding Workflow

### Traditional vs AI-Assisted

**Traditional:**
```
Think → Write → Test → Debug → Refactor → Ship
```

**AI-Assisted:**
```
Describe → Generate → Review → Refine → Test → Ship
```

### Key Principles

1. **Context is king** — The more context you provide, the better the output
2. **Iterate, don't accept first draft** — AI generates plausible code, not always correct
3. **Review every line** — You are responsible for what ships
4. **Run tests always** — Never assume AI code works
5. **Learn from AI** — Study the patterns it uses

## 2. Prompt Patterns for Code Generation

### Pattern 1: The CODE Framework

- **C**ontext: What project, stack, constraints
- **O**bjective: What you need
- **D**etails: Edge cases, performance requirements
- **E**xamples: Patterns to follow

In [ ]:
# Example: Using CODE framework to generate a function

# Bad prompt:
# "Write a function"

# Good prompt:
# "I'm building a FastAPI backend (Python 3.11, FastAPI 0.104).
# Write a function that validates email addresses.
# Requirements:
# - Use pydantic for validation
# - Support international emails
# - Return ValidationError with clear messages
# - Include type hints
# Here's my existing User model: [paste model]"

# Example output from such a prompt:
from pydantic import BaseModel, EmailStr, validator
import re

class UserCreate(BaseModel):
    email: EmailStr
    name: str
    
    @validator('name')
    def validate_name(cls, v):
        if len(v.strip()) < 2:
            raise ValueError('Name must be at least 2 characters')
        if not re.match(r'^[a-zA-Z\s]+$', v):
            raise ValueError('Name must contain only letters and spaces')
        return v.strip()

print("Model created with validation")

### Pattern 2: Role + Constraint

Assign a specific role and constraint to the AI.

In [ ]:
# Example: Using role-based prompting

# Prompt:
# "You are a senior Python engineer reviewing code for production readiness.
# Review this function for:
# 1. Error handling
# 2. Type safety
# 3. Performance
# 4. Security
# Be critical. Don't just say it looks good."

# Example AI response (what you'd get back):
review_example = """
CRITICAL ISSUES:
1. No input validation — SQL injection risk in line 15
2. Missing timeout on requests.get() — can hang indefinitely
3. No error handling for JSON decode failures

WARNINGS:
4. Using print() instead of logging
5. Connection not closed in finally block

SUGGESTIONS:
6. Add type hints for better IDE support
7. Consider connection pooling for database calls
"""

print("AI Review Output:")
print(review_example)

### Pattern 3: Before/After Refactoring

Show the AI your messy code and ask for specific improvements.

In [ ]:
# BEFORE: Messy code
def process(data):
    result = []
    for i in data:
        if i > 0:
            if i % 2 == 0:
                result.append(i * 2)
            else:
                result.append(i * 3)
        else:
            if i < -10:
                result.append(0)
            else:
                result.append(abs(i))
    return result

print("Before:", process([-5, 3, 4, -15, 2]))

# AFTER: AI-refactored version
from typing import List

def transform_positive(n: int) -> int:
    """Double even positive numbers, triple odd positive numbers."""
    return n * 2 if n % 2 == 0 else n * 3

def transform_negative(n: int) -> int:
    """Return 0 for very negative numbers, absolute value otherwise."""
    return 0 if n < -10 else abs(n)

def process(data: List[int]) -> List[int]:
    """Transform numbers based on their sign and parity."""
    return [
        transform_positive(n) if n > 0 else transform_negative(n)
        for n in data
    ]

print("After:", process([-5, 3, 4, -15, 2]))
print("\nImprovements:")
print("- Extracted helper functions")
print("- Added type hints")
print("- Added docstrings")
print("- Used list comprehension")
print("- Single responsibility per function")

## 3. Debugging Workflow with AI

### The Debugging Prompt Template

```
I'm getting this error:
[paste FULL traceback]

Here's the relevant code:
[paste the function/file]

What's the root cause? How do I fix it?
How do I prevent this class of error in the future?
```



In [ ]:
# Example: Debugging a common Python error

# The buggy code:
def get_user(user_id):
    users = {1: {"name": "Alice"}, 2: {"name": "Bob"}}
    return users[user_id]["email"]  # KeyError!

# The error:
# Traceback (most recent call last):
#   File "app.py", line 10, in <module>
#     print(get_user(1))
#   File "app.py", line 4, in get_user
#     return users[user_id]["email"]
# KeyError: 'email'

# AI would suggest:
from typing import Optional, Dict, Any

def get_user(user_id: int) -> Optional[Dict[str, Any]]:
    """Get user by ID. Returns None if not found."""
    users = {1: {"name": "Alice", "email": "alice@test.com"}, 
             2: {"name": "Bob", "email": "bob@test.com"}}
    
    user = users.get(user_id)
    if user is None:
        return None
    
    return {"name": user["name"], "email": user.get("email", "N/A")}

print("Fixed function handles missing keys gracefully")
print(get_user(1))  # Works
print(get_user(3))  # Returns None instead of crashing

## 4. Code Review with AI

### Review Prompt Template

```
Review this code for:
1. Correctness — does it do what it claims?
2. Security — any vulnerabilities?
3. Performance — any bottlenecks?
4. Readability — is it clear and maintainable?
5. Edge cases — what could go wrong?

Be specific. For each issue, provide:
- The problem
- Why it matters
- How to fix it
```

In [ ]:
# Example: Code that needs review
import sqlite3

def search_users(query):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    cursor.execute(f"SELECT * FROM users WHERE name LIKE '%{query}%'")
    results = cursor.fetchall()
    conn.close()
    return results

# AI Review would identify:
issues = """
SECURITY (Critical):
1. SQL INJECTION: f-string in SQL query allows injection attack
   Fix: Use parameterized queries:
   cursor.execute("SELECT * FROM users WHERE name LIKE ?", (f"%{query}%",))

RELIABILITY:
2. No error handling — if DB connection fails, crashes
3. Connection not closed in finally/with block
4. No timeout on database operations

PERFORMANCE:
5. LIKE '%query%' forces full table scan — consider full-text search
6. fetchall() loads all results into memory

READABILITY:
7. No type hints
8. No docstring
9. No logging for debugging
"""

print("AI Code Review:")
print(issues)

# Fixed version:
import sqlite3
from typing import List, Tuple
import logging

logger = logging.getLogger(__name__)

def search_users(query: str, db_path: str = 'users.db') -> List[Tuple]:
    """Search users by name with SQL injection protection."""
    if not query or not query.strip():
        raise ValueError("Search query cannot be empty")
    
    try:
        with sqlite3.connect(db_path, timeout=10) as conn:
            cursor = conn.cursor()
            cursor.execute(
                "SELECT * FROM users WHERE name LIKE ?",
                (f"%{query}%",)
            )
            results = cursor.fetchall()
            logger.info(f"Found {len(results)} users for query: {query}")
            return results
    except sqlite3.Error as e:
        logger.error(f"Database error: {e}")
        raise

print("\nFixed version applies all AI suggestions")

## 5. Iterative Development with AI

### The Refinement Loop

1. **Generate** — Get initial code from AI
2. **Run** — Execute and check output
3. **Identify issues** — What's missing or wrong?
4. **Refine** — Ask AI to fix specific issues
5. **Repeat** until satisfied

In [ ]:
# Iterative development example

# Generation 1: Basic function
def read_csv(filename):
    import csv
    with open(filename) as f:
        return list(csv.reader(f))

# After running, we discover issues:
# - No error handling for missing files
# - No encoding handling
# - Returns list of lists, not dicts

# Generation 2: Add error handling
import csv
from pathlib import Path
from typing import List, Dict, Any

def read_csv(filename: str, encoding: str = 'utf-8') -> List[Dict[str, str]]:
    """Read CSV file and return list of dictionaries."""
    filepath = Path(filename)
    
    if not filepath.exists():
        raise FileNotFoundError(f"CSV file not found: {filename}")
    
    if not filepath.suffix.lower() == '.csv':
        raise ValueError(f"File must be CSV, got: {filepath.suffix}")
    
    try:
        with open(filepath, encoding=encoding) as f:
            reader = csv.DictReader(f)
            return list(reader)
    except UnicodeDecodeError:
        raise ValueError(f"Cannot read file with encoding {encoding}")

# Generation 3: Add features
def read_csv(
    filename: str,
    encoding: str = 'utf-8',
    max_rows: int = None
) -> List[Dict[str, str]]:
    """Read CSV file with optional row limit."""
    filepath = Path(filename)
    
    if not filepath.exists():
        raise FileNotFoundError(f"CSV file not found: {filename}")
    
    with open(filepath, encoding=encoding) as f:
        reader = csv.DictReader(f)
        if max_rows:
            return [row for i, row in enumerate(reader) if i < max_rows]
        return list(reader)

print("Iterative improvement demonstrated")
print("Gen 1: Basic -> Gen 2: Error handling -> Gen 3: Features")

## 6. Key Takeaways

### Effective AI Coding Checklist

- [ ] Provide context (project, stack, constraints)
- [ ] Be specific about what you need
- [ ] Review every line of generated code
- [ ] Run tests after AI changes
- [ ] Iterate — first draft is rarely final
- [ ] Learn from the patterns AI uses
- [ ] Use version control before AI changes

### When AI Excels

- Boilerplate code (CRUD, configs, tests)
- Debugging (parsing errors, finding root causes)
- Refactoring (structural changes, pattern application)
- Documentation (docstrings, READMEs)
- Code review (identifying issues)

### When to Rely on Human Judgment

- Architecture decisions
- Security-critical code
- Business logic correctness
- Performance-critical paths
- Final code review